# Aula 1, O que é um agente e o agent loop

Notebook executável que acompanha a aula [01-agente-e-agent-loop.md](../../lessons/modulo-10-agentes/01-agente-e-agent-loop.md).

## O que vamos fazer aqui

Um agente é um LLM dentro de um ciclo de perceber, decidir, agir e observar. Vamos construir um
agente mínimo que decide entre uma calculadora segura e uma busca, com um controlador por regras.
Python puro.

## Ferramentas e agent loop

A calculadora avalia expressões com segurança (sem eval). O agente percebe a pergunta, decide a
ferramenta, age e observa o resultado.

In [ ]:
import re
import ast
import operator

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def calcular(expr):
    """Avalia uma expressão aritmética com segurança, sem usar eval."""
    def ev(no):
        if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
            return no.value
        if isinstance(no, ast.BinOp) and type(no.op) in OPS:
            return OPS[type(no.op)](ev(no.left), ev(no.right))
        if isinstance(no, ast.UnaryOp) and type(no.op) in OPS:
            return OPS[type(no.op)](ev(no.operand))
        raise ValueError("expressão não permitida")
    return ev(ast.parse(expr, mode="eval").body)


def buscar(consulta):
    base = {
        "derivada": "A derivada mede a taxa de variação de uma função.",
        "matriz": "Uma matriz organiza números em linhas e colunas.",
    }
    for chave, texto in base.items():
        if chave in consulta.lower():
            return texto
    return "Não encontrei no material."


def extrair_expressao(texto):
    return "".join(re.findall(r"\d+\.?\d*|[+\-*/()]", texto))


def agente(pergunta):
    if re.search(r"\d", pergunta) and re.search(r"[+\-*/]", pergunta):
        acao, arg = "calcular", extrair_expressao(pergunta)
    else:
        acao, arg = "buscar", pergunta
    observacao = calcular(arg) if acao == "calcular" else buscar(arg)
    return {"acao": acao, "argumento": arg, "observacao": observacao}


for p in ["quanto é 28*3/4 ?", "o que é a derivada?", "explique uma matriz"]:
    print(p, "->", agente(p))

O agente roteia a conta para a calculadora (28*3/4 = 21.0) e as perguntas de conteúdo
para a busca. Cada chamada percorre o ciclo: percebe, decide, age e observa. É o esqueleto de todo
agente. Para o projeto, acrescente uma terceira ferramenta.